In [ ]:
#mengimport google colabs ke gdrive
from google.colab import drive
drive.mount('/content/gdrive')

In [ ]:

#Import library
import pandas as pd # module yang digunakan untuk analisis data, editing dataframe
import numpy as np # module yang digunakan untuk operasi matematika
import matplotlib as plt # module yang digunakan untuk melakukan visualisasi data
import seaborn as sns # module yang digunakan untuk melakukan visualisasi data
import os # -> untuk melakukan operasi sistem di google colab



In [ ]:
os.chdir('/content/gdrive/MyDrive/Capstone Projek Dataset')

In [ ]:
os.listdir()

In [ ]:
# Membaca file CSV
df = pd.read_csv("flight.csv")

# Understanding Data

In [ ]:
df.head(15)

In [ ]:
df.shape

In [ ]:
df.columns

In [ ]:
#hapus kolom dengan NaN diatas 70%
threshold = 0.7

#menghitung persentase nilai NaN di setiap kolom
nan_percentage = df.isna().mean()

#Menghitung kolom yang persentase nilai NaN nya di atas batas
columns_to_drop = nan_percentage[nan_percentage > threshold].index
df_cleaned = df.drop(columns=columns_to_drop)

print("kolom yang dihapus:", columns_to_drop)

In [ ]:
df = df_cleaned

In [ ]:
df.shape

In [ ]:
df.columns

In [ ]:
ticket_price_unique = df['Ticket prize(Doller)'].unique()
ticket_price_unique

In [ ]:
airline_unique = df['Airline name '].unique()
airline_unique

In [ ]:
# Menghitung jumlah tiket terjual berdasarkan maskapai penerbangan
jumlah_tiket_terjual = df.groupby('Airline name ')['Ticket prize(Doller)'].count().reset_index()

jumlah_tiket_terjual.columns = ['Airline name ', 'jumlah tiket terjual']

# Merge hasil perhitungan ke dataframe
df = pd.merge(df, jumlah_tiket_terjual, on='Airline name ', how='left')

# Tampilkan dataframe dengan kolom baru
print(df)

In [ ]:
df.columns

In [ ]:
df.head(15)

##Data Cleaning

In [ ]:
df.duplicated().sum()

In [ ]:
#menghapus salah satu data yang duplikat
df = df.drop_duplicates( keep='last')
df.duplicated().sum()

In [ ]:
#cek missing values
df.isna().sum()

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
#checking if columns have null values
df.isna().any()

In [ ]:
#copy data
df2 = df.copy()

In [ ]:
# Mengganti nilai yang hilang pada df2
df2['jumlah tiket terjual'].fillna(method='bfill', inplace=True)  # Mengisi nilai yang hilang dengan nilai setelahnya

# Mengecek data yang hilang pada df2
df2['jumlah tiket terjual'].isna().sum()

In [ ]:
#cek missing values
df2.isna().sum()

##EDA

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


# Preprocess categorical variables (one-hot encoding)
data = pd.get_dummies(df2, columns=['Airline name ', 'Depreture  Airport', '1st Stoppage', 'Destination Airport'])

# 1. Deskripsi Statistik
ticket_sales_stats = data['jumlah tiket terjual'].describe()
print(ticket_sales_stats)

# 2. Distribusi
plt.figure(figsize=(10, 6))
sns.histplot(data['jumlah tiket terjual'], kde=True)
plt.title('Distribusi Jumlah Tiket Terjual')
plt.xlabel('Jumlah Tiket Terjual')
plt.ylabel('Frekuensi')
plt.show()

# 3. Korelasi

# Remove non-numeric columns for correlation analysis
numeric_data = df2.select_dtypes(include=[np.number])

correlation_matrix = numeric_data.corr()
plt.figure(figsize=(10, 6))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm')
plt.title('Korelasi antara Variabel')
plt.show()

# 4. Pengaruh Variabel
# Anda mungkin perlu menyesuaikan visualisasi ini tergantung pada jumlah maskapai dan bandara
plt.figure(figsize=(10, 6))
sns.boxplot(x='Airline name ', y='jumlah tiket terjual', data=df2)
plt.title('Pengaruh Maskapai terhadap Jumlah Tiket Terjual')
plt.xlabel('Maskapai')
plt.ylabel('Jumlah Tiket Terjual')
plt.xticks(rotation=45)
plt.show()



In [ ]:
plt.hist(df2['jumlah tiket terjual'], bins=20, color='skyblue', edgecolor='black')
plt.xlabel('Jumlah Tiket Terjual')
plt.ylabel('Frequency')
plt.title('Histogram of Jumlah Tiket Terjual')
plt.show()

In [ ]:
df2.head()

In [ ]:
# @title Arrival Date

from matplotlib import pyplot as plt
import seaborn as sns
df2.groupby('Arrival Date').size().plot(kind='barh', color=sns.palettes.mpl_palette('Dark2'))
plt.gca().spines[['top', 'right',]].set_visible(False)

In [ ]:
# @title Arrival Date vs jumlah tiket terjual

from matplotlib import pyplot as plt
import seaborn as sns
figsize = (12, 1.2 * len(df2['Arrival Date'].unique()))
plt.figure(figsize=figsize)
sns.violinplot(df2, x='jumlah tiket terjual', y='Arrival Date', inner='box', palette='Dark2')
sns.despine(top=True, right=True, bottom=True, left=True)

In [ ]:
#membuat pivot table
statewise_data = pd.pivot_table(df2, values=['jumlah tiket terjual'], index="Airline name ", aggfunc=max)
statewise_data=statewise_data.sort_values(by="jumlah tiket terjual",ascending=False)
#menambahkan gradien warna pada dataframe
statewise_data.style.background_gradient(cmap="CMRmap")

In [ ]:
import matplotlib.pyplot as plt

# Kelompokkan data berdasarkan maskapai dan total tiket terjual
jumlah_tiket_terjual_maskapai = df2.groupby('Airline name ')['jumlah tiket terjual'].sum().reset_index()

# Urutkan secara menurun
top_10_airline = jumlah_tiket_terjual_maskapai.sort_values(by='jumlah tiket terjual', ascending=False).head(10)

# Buat plot batang
plt.figure(figsize=(15, 10))
plt.bar(top_10_airline['Airline name '], top_10_airline['jumlah tiket terjual'], color='blue')

# Tambahkan judul, label sumbu, dan rotasi label x
plt.title('Top 10 maskapai')
plt.xlabel('Airline name ')
plt.ylabel('jumlah tiket terjual')
plt.xticks(rotation=45, ha='right')  # Rotasi label x ke kanan

# Tambahkan grid pada sumbu y
plt.grid(axis='y', linestyle='--', alpha=0.7)

# Tampilkan plot
plt.tight_layout()
plt.show()

In [ ]:
df2.to_csv('df2', index=False)